# **Taller de Aprendizaje Automático**
## **Resolución del Problema 4**

---

### **Información del Estudiante**
* **Nombre del Autor:** Breadley Josshuen Marin Velasco y David López López 
* **Institución Educativa:** Universidad Nacional de Colombia
* **Curso / Materia:** Aprendizaje Automático (Machine Learning)
* **Fecha:** Octubre, 2026

---

### **Requerimientos y Descripción Técnica del Problema 4**

En esta sección se detallan e implementan en `NumPy` diversos algoritmos fundamentales de aprendizaje supervisado, garantizando soluciones de alta estabilidad numérica, manejo robusto de excepciones e instrumentación detallada del proceso de optimización:

1. **Mínimos Cuadrados Ordinarios (OLS) y Regresión Ridge:**
   * Solución exacta por sistema de ecuaciones normales resuelto de manera estable.
   * Penalización $L_2$ aplicada estrictamente sobre los coeficientes de características $w$, manteniendo el intercepto $b$ **sin penalizar**.

2. **Descenso por Gradiente para Regresión Lineal y Logística:**
   * Optimización interactiva basada en gradientes analíticos con opción de penalización $L_2$ regularizada.
   * Mitigación de desbordamiento (*overflow/underflow*) en la función de activación sigmoide y el cómputo de la pérdida de entropía cruzada mediante formulaciones estables.

3. **Gradiente Proximal para Lasso Lineal:**
   * Implementación rigurosa utilizando el operador de umbralización suave (*Soft-Thresholding*) en cada paso proximal de forma iterativa, evitando enfoques alternativos e incorrectos como redondear o truncar coeficientes al final de la optimización.
   * Criterio de convergencia formal basado en la norma del residuo proximal completo: $\|\theta^+ - \theta\|_2 / \eta$, incluyendo el intercepto sin penalizar.

4. **Robustez, Validación e Instrumentación:**
   * Monitoreo, registro e historial de la función objetivo completa en cada paso.
   * Control de criterios de parada basados en tolerancia física del gradiente o residuo proximal, paso de aprendizaje (tasa de aprendizaje $\eta$) y límite máximo de iteraciones.
   * Validación rigurosa de dimensiones compatibles, entradas finitas (control de NaN/Inf) e hiperparámetros no negativos.
   * Reinicialización por defecto de los parámetros en sucesivas llamadas de `.fit()`, soportando también la opción de arranque en caliente (*warm start*).

# Sección 1: Implementación de Modelos desde Cero en NumPy

In [4]:
# 1. Funciones matemáticas auxiliares (Sigmoide estable y Soft-Thresholding)
import numpy as np

def sigmoid_stable(z):
    z = np.asarray(z, dtype=np.float64)
    pos_mask = z >= 0
    neg_mask = ~pos_mask

    res = np.empty_like(z)
    res[pos_mask] = 1.0 / (1.0 + np.exp(-z[pos_mask]))
    exp_z = np.exp(z[neg_mask])
    res[neg_mask] = exp_z / (1.0 + exp_z)
    return res

def soft_thresholding(v, gamma):
    return np.sign(v) * np.maximum(0.0, np.abs(v) - gamma)

In [5]:
# 2. Clase base para modelos lineales e instrumentación
class BaseLinearModelNumPy:
    def __init__(self, warm_start=False):
        self.warm_start = warm_start
        self.w_ = None
        self.b_ = None
        self.n_iter_ = 0
        self.termination_reason_ = None
        self.history_obj_ = []

    def _validate_inputs(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        if not np.all(np.isfinite(X)):
            raise ValueError("La matriz X contiene entradas no finitas (NaN o Inf).")

        if y is not None:
            y = np.asarray(y, dtype=np.float64)
            if not np.all(np.isfinite(y)):
                raise ValueError("El vector y contiene entradas no finitas (NaN o Inf).")
            if X.shape[0] != y.shape[0]:
                raise ValueError(f"Dimensiones incompatibles: X tiene {X.shape[0]} filas y y tiene {y.shape[0]}.")
            return X, y
        return X

    def _init_params(self, n_features):
        if not self.warm_start or self.w_ is None or len(self.w_) != n_features:
            self.w_ = np.zeros(n_features, dtype=np.float64)
            self.b_ = 0.0
            self.history_obj_ = []
            self.n_iter_ = 0
            self.termination_reason_ = None

In [6]:
# 3. Mínimos Cuadrados Ordinarios (OLS) y Ridge (Solución analítica)
class LinearRegressionClosedFormNumPy(BaseLinearModelNumPy):
    def __init__(self, l2_reg=0.0):
        super().__init__(warm_start=False)
        if l2_reg < 0:
            raise ValueError("El hiperparámetro l2_reg (lambda_2) debe ser >= 0.")
        self.l2_reg = float(l2_reg)

    def fit(self, X, y):
        X, y = self._validate_inputs(X, y)
        N, D = X.shape

        X_tilde = np.hstack([np.ones((N, 1)), X])
        P = np.eye(D + 1)
        P[0, 0] = 0.0  # Intercepto no penalizado

        A = X_tilde.T @ X_tilde + N * self.l2_reg * P
        b_vec = X_tilde.T @ y

        theta, _, _, _ = np.linalg.lstsq(A, b_vec, rcond=None)

        self.b_ = float(theta[0])
        self.w_ = theta[1:]

        z = X @ self.w_ + self.b_
        j_reg = (1.0 / (2.0 * N)) * np.sum((z - y) ** 2)
        obj_val = j_reg + (self.l2_reg / 2.0) * np.sum(self.w_ ** 2)

        self.history_obj_ = [obj_val]
        self.n_iter_ = 1
        self.termination_reason_ = "closed_form_exact"
        return self

    def predict(self, X):
        X = self._validate_inputs(X)
        return X @ self.w_ + self.b_

In [7]:
# 4. Descenso por gradiente (Regresión Lineal y Logística con L2)
class GradientDescentLinearNumPy(BaseLinearModelNumPy):
    def __init__(self, loss_type="linear", l2_reg=0.0, eta=0.1, max_iter=2000, tol=1e-6, warm_start=False):
        super().__init__(warm_start=warm_start)
        if loss_type not in ["linear", "logistic"]:
            raise ValueError("loss_type debe ser 'linear' o 'logistic'.")
        if l2_reg < 0 or eta <= 0 or tol < 0:
            raise ValueError("Asegúrese de que l2_reg >= 0, eta > 0 y tol >= 0.")

        self.loss_type = loss_type
        self.l2_reg = float(l2_reg)
        self.eta = float(eta)
        self.max_iter = int(max_iter)
        self.tol = float(tol)

    def _compute_loss_and_grad(self, X, y, w, b):
        N = X.shape[0]
        z = X @ w + b

        if self.loss_type == "linear":
            res = z - y
            j_loss = (1.0 / (2.0 * N)) * np.sum(res ** 2)
            grad_w = (1.0 / N) * (X.T @ res) + self.l2_reg * w
            grad_b = (1.0 / N) * np.sum(res)

        elif self.loss_type == "logistic":
            p = sigmoid_stable(z)
            loss_i = np.maximum(0.0, z) - y * z + np.log(1.0 + np.exp(-np.abs(z)))
            j_loss = np.mean(loss_i)

            err = p - y
            grad_w = (1.0 / N) * (X.T @ err) + self.l2_reg * w
            grad_b = (1.0 / N) * np.sum(err)

        obj_val = j_loss + (self.l2_reg / 2.0) * np.sum(w ** 2)
        return obj_val, grad_w, grad_b

    def fit(self, X, y):
        X, y = self._validate_inputs(X, y)
        N, D = X.shape
        self._init_params(D)

        for k in range(1, self.max_iter + 1):
            obj, grad_w, grad_b = self._compute_loss_and_grad(X, y, self.w_, self.b_)
            self.history_obj_.append(obj)

            grad_norm = np.sqrt(np.sum(grad_w ** 2) + grad_b ** 2)
            if grad_norm < self.tol:
                self.termination_reason_ = "tol_reached"
                self.n_iter_ = k
                break

            self.w_ -= self.eta * grad_w
            self.b_ -= self.eta * grad_b

        if self.termination_reason_ is None:
            self.termination_reason_ = "max_iter"
            self.n_iter_ = self.max_iter

        return self

    def predict_proba(self, X):
        if self.loss_type != "logistic":
            raise AttributeError("predict_proba solo aplica para loss_type='logistic'.")
        X = self._validate_inputs(X)
        z = X @ self.w_ + self.b_
        p1 = sigmoid_stable(z)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X):
        X = self._validate_inputs(X)
        z = X @ self.w_ + self.b_
        if self.loss_type == "linear":
            return z
        else:
            return (sigmoid_stable(z) >= 0.5).astype(int)

In [8]:
# 5. Gradiente Proximal para Lasso
class LassoProximalNumPy(BaseLinearModelNumPy):
    def __init__(self, l1_reg=0.1, eta=0.01, max_iter=5000, tol=1e-6, warm_start=False):
        super().__init__(warm_start=warm_start)
        if l1_reg < 0 or eta <= 0 or tol < 0:
            raise ValueError("Asegúrese de que l1_reg >= 0, eta > 0 y tol >= 0.")

        self.l1_reg = float(l1_reg)
        self.eta = float(eta)
        self.max_iter = int(max_iter)
        self.tol = float(tol)

    def fit(self, X, y):
        X, y = self._validate_inputs(X, y)
        N, D = X.shape
        self._init_params(D)

        for k in range(1, self.max_iter + 1):
            z = X @ self.w_ + self.b_
            res = z - y
            j_reg = (1.0 / (2.0 * N)) * np.sum(res ** 2)
            obj_val = j_reg + self.l1_reg * np.sum(np.abs(self.w_))
            self.history_obj_.append(obj_val)

            grad_w_smooth = (1.0 / N) * (X.T @ res)
            grad_b_smooth = (1.0 / N) * np.sum(res)

            b_next = self.b_ - self.eta * grad_b_smooth
            w_intermediate = self.w_ - self.eta * grad_w_smooth
            w_next = soft_thresholding(w_intermediate, self.eta * self.l1_reg)

            diff_w = w_next - self.w_
            diff_b = b_next - self.b_
            prox_residual = np.sqrt(np.sum(diff_w ** 2) + diff_b ** 2) / self.eta

            self.w_ = w_next
            self.b_ = b_next

            if prox_residual < self.tol:
                self.termination_reason_ = "tol_reached"
                self.n_iter_ = k
                break

        if self.termination_reason_ is None:
            self.termination_reason_ = "max_iter"
            self.n_iter_ = self.max_iter

        return self

    def predict(self, X):
        X = self._validate_inputs(X)
        return X @ self.w_ + self.b_

# Sección 2: Suite de Verificación Numérica y Pruebas Automáticas (Problema 4b)

Esta sección implementa pruebas automáticas para auditar rigurosamente la precisión numérica, la estabilidad frente a desbordamientos y el cumplimiento de las condiciones de optimalidad de los modelos implementados.

In [9]:
# ==============================================================================
# PRUEBA 1: VERIFICACIÓN DE GRADIENTES POR DIFERENCIAS CENTRALES
# ==============================================================================

def check_gradients_central_diff(loss_type="linear", l2_reg=0.1, n_points=3, h=1e-6, seed=2026):
    """
    Verifica los gradientes analíticos respecto a (w, b) usando diferencias centrales.
    """
    np.random.seed(seed)
    N, D = 50, 5
    X = np.random.randn(N, D)

    if loss_type == "linear":
        y = X @ np.random.randn(D) + np.random.randn(N) * 0.5
    else:
        y = (np.random.randn(N) > 0).astype(np.float64)

    model = GradientDescentLinearNumPy(loss_type=loss_type, l2_reg=l2_reg)

    print(f"--- VERIFICACIÓN DE GRADIENTES: {loss_type.upper()} (l2_reg={l2_reg}) ---")

    for point_idx in range(1, n_points + 1):
        # Punto aleatorio moderado de evaluación theta = [b, w_1, ..., w_D]
        w_curr = np.random.randn(D) * 0.5
        b_curr = float(np.random.randn(1) * 0.5)

        # 1. Gradiente Analítico
        obj_base, grad_w_anal, grad_b_anal = model._compute_loss_and_grad(X, y, w_curr, b_curr)
        g_anal = np.hstack([grad_b_anal, grad_w_anal])

        # 2. Gradiente Numérico por Diferencias Centrales
        g_num = np.zeros_like(g_anal)

        # Perturbación para el intercepto b
        obj_plus, _, _ = model._compute_loss_and_grad(X, y, w_curr, b_curr + h)
        obj_minus, _, _ = model._compute_loss_and_grad(X, y, w_curr, b_curr - h)
        g_num[0] = (obj_plus - obj_minus) / (2.0 * h)

        # Perturbación para los pesos w_j
        for j in range(D):
            w_plus = w_curr.copy()
            w_minus = w_curr.copy()
            w_plus[j] += h
            w_minus[j] -= h

            obj_p, _, _ = model._compute_loss_and_grad(X, y, w_plus, b_curr)
            obj_m, _, _ = model._compute_loss_and_grad(X, y, w_minus, b_curr)
            g_num[j + 1] = (obj_p - obj_m) / (2.0 * h)

        # 3. Cálculo del Error Relativo \delta_g
        norm_diff = np.linalg.norm(g_anal - g_num)
        norm_anal = np.linalg.norm(g_anal)
        delta_g = norm_diff / max(1.0, norm_anal, np.linalg.norm(g_num))

        status = "PASÓ (delta_g < 1e-5)" if delta_g < 1e-5 else "FALLÓ"
        print(f"  Punto {point_idx}: ||g_anal||_2 = {norm_anal:.6f} | delta_g = {delta_g:.2e} -> {status}")

# Ejecutar verificación para ambos tipos de pérdida
check_gradients_central_diff(loss_type="linear", l2_reg=0.1)
print()
check_gradients_central_diff(loss_type="logistic", l2_reg=0.5)

--- VERIFICACIÓN DE GRADIENTES: LINEAR (l2_reg=0.1) ---
  Punto 1: ||g_anal||_2 = 1.288748 | delta_g = 3.80e-10 -> PASÓ (delta_g < 1e-5)
  Punto 2: ||g_anal||_2 = 2.600504 | delta_g = 3.19e-10 -> PASÓ (delta_g < 1e-5)
  Punto 3: ||g_anal||_2 = 2.295937 | delta_g = 2.20e-10 -> PASÓ (delta_g < 1e-5)

--- VERIFICACIÓN DE GRADIENTES: LOGISTIC (l2_reg=0.5) ---
  Punto 1: ||g_anal||_2 = 0.766928 | delta_g = 1.58e-10 -> PASÓ (delta_g < 1e-5)
  Punto 2: ||g_anal||_2 = 0.593529 | delta_g = 2.11e-10 -> PASÓ (delta_g < 1e-5)
  Punto 3: ||g_anal||_2 = 0.682961 | delta_g = 2.18e-10 -> PASÓ (delta_g < 1e-5)


/tmp/ipykernel_2904/701536729.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  b_curr = float(np.random.randn(1) * 0.5)


In [10]:
# ==============================================================================
# PRUEBA 2: EVALUACIÓN FINITA DE SIGMOIDE Y PÉRDIDA LOGÍSTICA
# ==============================================================================

def test_sigmoid_and_loss_finiteness():
    z_values = np.array([-1000.0, -100.0, -1.0, 0.0, 1.0, 100.0, 1000.0])

    print("--- PRUEBA DE EVALUACIÓN FINITA: SIGMOIDE Y PÉRDIDA LOGÍSTICA ---")
    print(f"{'z':>8} | {'y':>3} | {'p_1':>12} | {'p_0':>12} | {'Suma Prob':>10} | {'Loss (LSE)':>12} | Status")
    print("-" * 75)

    all_passed = True
    for z in z_values:
        p1 = float(sigmoid_stable(z))
        p0 = 1.0 - p1
        prob_sum = p0 + p1

        for y in [0.0, 1.0]:
            # Pérdida estable mediante Log-Sum-Exp
            loss_val = float(np.maximum(0.0, z) - y * z + np.log(1.0 + np.exp(-np.abs(z))))

            is_finite = np.isfinite(p1) and np.isfinite(loss_val)
            probs_valid = (0.0 <= p1 <= 1.0) and (0.0 <= p0 <= 1.0) and np.isclose(prob_sum, 1.0)

            passed = is_finite and probs_valid
            if not passed:
                all_passed = False

            status_str = "OK" if passed else "ERROR"
            print(f"{z:8.1f} | {int(y):3d} | {p1:12.6e} | {p0:12.6e} | {prob_sum:10.6f} | {loss_val:12.6f} | {status_str}")

    print("-" * 75)
    print(f"Resultado General: {'TODAS LAS PRUEBAS PASARON CORRECTAMENTE' if all_passed else 'FALLO EN ALGUNAS PRUEBAS'}")

test_sigmoid_and_loss_finiteness()

--- PRUEBA DE EVALUACIÓN FINITA: SIGMOIDE Y PÉRDIDA LOGÍSTICA ---
       z |   y |          p_1 |          p_0 |  Suma Prob |   Loss (LSE) | Status
---------------------------------------------------------------------------
 -1000.0 |   0 | 0.000000e+00 | 1.000000e+00 |   1.000000 |     0.000000 | OK
 -1000.0 |   1 | 0.000000e+00 | 1.000000e+00 |   1.000000 |  1000.000000 | OK
  -100.0 |   0 | 3.720076e-44 | 1.000000e+00 |   1.000000 |     0.000000 | OK
  -100.0 |   1 | 3.720076e-44 | 1.000000e+00 |   1.000000 |   100.000000 | OK
    -1.0 |   0 | 2.689414e-01 | 7.310586e-01 |   1.000000 |     0.313262 | OK
    -1.0 |   1 | 2.689414e-01 | 7.310586e-01 |   1.000000 |     1.313262 | OK
     0.0 |   0 | 5.000000e-01 | 5.000000e-01 |   1.000000 |     0.693147 | OK
     0.0 |   1 | 5.000000e-01 | 5.000000e-01 |   1.000000 |     0.693147 | OK
     1.0 |   0 | 7.310586e-01 | 2.689414e-01 |   1.000000 |     1.313262 | OK
     1.0 |   1 | 7.310586e-01 | 2.689414e-01 |   1.000000 |     0.313262 |

In [19]:
# ==============================================================================
# PRUEBA 3: RESIDUOS DE ECUACIONES NORMALES Y CONDICIONES DE OPTIMALIDAD
# ==============================================================================

def check_lasso_kkt_conditions(X, y, w_hat, b_hat, l1_reg, tol_kkt=1e-3):
    """
    Verifica la condicion de optimalidad coordenada a coordenada para Lasso
    (ver Problema 2b): para cada j,
        - si w_hat[j] > 0:  (1/N) x_j^T r = -lambda1          (r = y - X w - b)
        - si w_hat[j] < 0:  (1/N) x_j^T r =  lambda1
        - si w_hat[j] = 0:  |(1/N) x_j^T r| <= lambda1
    y para el intercepto (sin penalizar): (1/N) 1_N^T r = 0.
    Retorna el maximo incumplimiento (violacion) de estas condiciones.
    """
    N, D = X.shape
    r = y - (X @ w_hat + b_hat)
    corr = (X.T @ r) / N  # (1/N) x_j^T r, para cada coordenada j

    violations = np.zeros(D)
    for j in range(D):
        if w_hat[j] > 1e-8:
            violations[j] = np.abs(corr[j] + l1_reg)
        elif w_hat[j] < -1e-8:
            violations[j] = np.abs(corr[j] - l1_reg)
        else:  # w_hat[j] == 0 (dentro de tolerancia numerica)
            violations[j] = max(0.0, np.abs(corr[j]) - l1_reg)

    # Condicion del intercepto, sin penalizar: debe ser estacionario exacto
    intercept_violation = np.abs(np.sum(r) / N)

    max_violation = max(np.max(violations), intercept_violation)
    return max_violation, violations, intercept_violation


def test_optimality_conditions(seed=2026):
    np.random.seed(seed)
    N, D = 100, 4
    X = np.random.randn(N, D)
    y = X @ np.array([2.0, -1.0, 0.5, 0.0]) + np.random.randn(N) * 0.1

    print("--- PRUEBA 3: RESIDUOS DE ECUACIONES NORMALES Y OPTIMALIDAD ---")

    # 1. OLS con columna duplicada (x_2 = x_1)
    X_dup = X.copy()
    X_dup[:, 1] = X_dup[:, 0]  # Duplicar columna 0 en columna 1
    ols_model = LinearRegressionClosedFormNumPy(l2_reg=0.0).fit(X_dup, y)

    X_tilde_dup = np.hstack([np.ones((N, 1)), X_dup])
    r_ols = y - ols_model.predict(X_dup)
    residual_ols = np.linalg.norm(X_tilde_dup.T @ r_ols) / N
    print(f"1. OLS (columna duplicada) -> Residual ||X_tilde^T * r|| / N: {residual_ols:.2e} (Esperado < 1e-5)")

    # 2. Ridge con lambda_2 > 0
    l2_val = 0.5
    ridge_model = LinearRegressionClosedFormNumPy(l2_reg=l2_val).fit(X, y)
    X_tilde = np.hstack([np.ones((N, 1)), X])
    r_ridge = y - ridge_model.predict(X)
    theta_ridge = np.hstack([ridge_model.b_, ridge_model.w_])

    P = np.eye(D + 1)
    P[0, 0] = 0.0  # Intercepto no penalizado

    # Ecuación normal Ridge: -X_tilde^T * r + N * lambda_2 * P * theta = 0
    residual_ridge = np.linalg.norm(-X_tilde.T @ r_ridge + N * l2_val * (P @ theta_ridge)) / N
    print(f"2. Ridge (l2_reg={l2_val}) -> Residual Ecuaciones Normales: {residual_ridge:.2e} (Esperado < 1e-5)")

    # 3. Lasso con Ortogonalidad (X^T * X / N = I_D)
    Q, _ = np.linalg.qr(np.random.randn(N, D))
    X_orth = Q * np.sqrt(N)  # Garantiza X_orth^T * X_orth / N = I_D
    y_orth = X_orth @ np.array([3.0, 0.0, -1.5, 0.0]) + np.random.randn(N) * 0.05

    l1_val = 0.2
    lasso_orth = LassoProximalNumPy(l1_reg=l1_val, eta=0.01, max_iter=10000, tol=1e-7).fit(X_orth, y_orth)

    # Verificación de regla de Soft-Thresholding analítica vs aproximación del modelo
    s_empirico = (X_orth.T @ (y_orth - lasso_orth.b_)) / N
    w_teorico = soft_thresholding(s_empirico, l1_val)
    diff_lasso_orth = np.max(np.abs(lasso_orth.w_ - w_teorico))
    print(f"3. Lasso (matriz ortogonal) -> Max diferencia vs Soft-Thresholding teorico: {diff_lasso_orth:.2e} (Esperado < 1e-4)")

    # 4. Lasso sin Ortogonalidad -> Verificación de condición de optimalidad (KKT/subgradiente)
    lasso_general = LassoProximalNumPy(l1_reg=l1_val, eta=0.01, max_iter=10000, tol=1e-7).fit(X, y)

    max_violation, per_coord_violation, intercept_violation = check_lasso_kkt_conditions(
        X, y, lasso_general.w_, lasso_general.b_, l1_reg=l1_val
    )

    print(f"4. Lasso (matriz general) -> Motivo Terminacion: "
          f"'{lasso_general.termination_reason_}' en {lasso_general.n_iter_} iters")
    print(f"   Violacion maxima de condicion de optimalidad coordenada a coordenada: "
          f"{max_violation:.2e} (esperado < 1e-3)")
    print(f"   Violacion en la condicion del intercepto (1/N * 1^T r = 0): "
          f"{intercept_violation:.2e}")
    print(f"   Violacion por coordenada: {np.round(per_coord_violation, 6)}")

    status = "CUMPLIDO" if max_violation < 1e-3 else "REVISAR (posible convergencia incompleta)"
    print(f"   -> {status}")

test_optimality_conditions()

--- PRUEBA 3: RESIDUOS DE ECUACIONES NORMALES Y OPTIMALIDAD ---
1. OLS (columna duplicada) -> Residual ||X_tilde^T * r|| / N: 2.47e-15 (Esperado < 1e-5)
2. Ridge (l2_reg=0.5) -> Residual Ecuaciones Normales: 1.18e-15 (Esperado < 1e-5)
3. Lasso (matriz ortogonal) -> Max diferencia vs Soft-Thresholding teorico: 6.39e-08 (Esperado < 1e-4)
4. Lasso (matriz general) -> Motivo Terminacion: 'tol_reached' en 2115 iters
   Violacion maxima de condicion de optimalidad coordenada a coordenada: 4.00e-01 (esperado < 1e-3)
   Violacion en la condicion del intercepto (1/N * 1^T r = 0): 1.30e-08
   Violacion por coordenada: [0.4 0.4 0.4 0. ]
   -> REVISAR (posible convergencia incompleta)


In [12]:
# ==============================================================================
# PRUEBA 4: CONSISTENCIA DE PREDICCIONES Y PERSISTENCIA DE PARÁMETROS
# ==============================================================================

def test_persistence_and_predict_consistency(seed=2026):
    np.random.seed(seed)
    N, D = 40, 3
    X = np.random.randn(N, D)
    y = (np.random.randn(N) > 0).astype(np.float64)

    print("--- PRUEBA 4: CONSISTENCIA DE PREDICCIONES Y PERSISTENCIA ---")

    # 1. Entrenar modelo original de Regresión Logística
    model_orig = GradientDescentLinearNumPy(loss_type="logistic", l2_reg=0.1, eta=0.1, max_iter=1000)
    model_orig.fit(X, y)

    preds_orig = model_orig.predict(X)
    probs_orig = model_orig.predict_proba(X)

    # 2. Persistencia: Exportar parámetros (w, b) e inyectarlos en una nueva instancia
    saved_w = model_orig.w_.copy()
    saved_b = float(model_orig.b_)

    model_loaded = GradientDescentLinearNumPy(loss_type="logistic", l2_reg=0.1)
    model_loaded.w_ = saved_w
    model_loaded.b_ = saved_b

    preds_loaded = model_loaded.predict(X)
    probs_loaded = model_loaded.predict_proba(X)

    # 3. Verificaciones de Consistencia y Axiomas de Probabilidad
    diff_preds = np.max(np.abs(preds_orig - preds_loaded))
    diff_probs = np.max(np.abs(probs_orig - probs_loaded))

    probs_in_range = np.all((probs_loaded >= 0.0) & (probs_loaded <= 1.0))
    sum_is_one = np.allclose(np.sum(probs_loaded, axis=1), 1.0, atol=1e-12)

    print(f"Max diff en predicciones binarias: {diff_preds}")
    print(f"Max diff en probabilidades (predict_proba): {diff_probs:.2e}")
    print(f"Probabilidades en rango [0, 1]: {probs_in_range}")
    print(f"Suma de probabilidades por fila igual a 1.0: {sum_is_one}")

    passed = (diff_preds == 0) and (diff_probs < 1e-14) and probs_in_range and sum_is_one
    print(f"Resultado Prueba 4: {'PASÓ CORRECTAMENTE' if passed else 'FALLÓ'}")

test_persistence_and_predict_consistency()

--- PRUEBA 4: CONSISTENCIA DE PREDICCIONES Y PERSISTENCIA ---
Max diff en predicciones binarias: 0
Max diff en probabilidades (predict_proba): 0.00e+00
Probabilidades en rango [0, 1]: True
Suma de probabilidades por fila igual a 1.0: True
Resultado Prueba 4: PASÓ CORRECTAMENTE


# Sección 3: Contraste y Verificación contra scikit-learn (Problema 4c)

Para realizar una verificación empírica rigurosa, se establece la correspondencia matemática exacta entre los hiperparámetros de penalización de nuestras implementaciones ($\lambda_1, \lambda_2$) y los parámetros utilizados por las clases de `scikit-learn` (`alpha`, `C`).

### Tabla de Correspondencia y Equivalencia de Objetivos

| Modelo | Función Objetivo del Taller ($F(w,b)$) | Función Objetivo de `scikit-learn` | Equivalencia Directa | Dependencia de $N$ |
| :--- | :--- | :--- | :--- | :--- |
| **OLS** | $\frac{1}{2N} \Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2$ | $\frac{1}{2N} \Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2$ | N/A | Invariante |
| **Ridge** | $\frac{1}{2N} \Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2 + \frac{\lambda_2}{2} \Vert{}w\Vert{}_2^2$ | $\Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2 + \alpha_{\text{Ridge}} \Vert{}w\Vert{}_2^2$ | $\alpha_{\text{Ridge}} = N \cdot \lambda_2$ | Proporcional a $N$ |
| **Lasso** | $\frac{1}{2N} \Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2 + \lambda_1 \Vert{}w\Vert{}_1$ | $\frac{1}{2N} \Vert{}Xw + b\mathbf{1}_N - y\Vert{}_2^2 + \alpha_{\text{Lasso}} \Vert{}w\Vert{}_1$ | $\alpha_{\text{Lasso}} = \lambda_1$ | Invariante |
| **Regresión Logística** | $J_{\text{log}}(w,b) + \frac{\lambda_2}{2} \Vert{}w\Vert{}_2^2$ | $C \sum_{n=1}^N \ell_n + \frac{1}{2} \Vert{}w\Vert{}_2^2$ | $C = \frac{1}{N \cdot \lambda_2}$ | Inversamente proporcional a $N$ |

*Nota sobre la normalización:*
- En **Ridge**, `scikit-learn` no divide el término de ajuste residual por $2N$, sino que suma la norma residual no escalada. Por ende, para igualar la penalización por muestra $\lambda_2$, debe escalarse `alpha` por el número de observaciones de entrenamiento: $\alpha_{\text{Ridge}} = N \lambda_2$[cite: 13, 15].
- En **Lasso**, `scikit-learn` normaliza la pérdida por $2N$, por lo que $\alpha_{\text{Lasso}}$ es directamente idéntico a $\lambda_1$[cite: 13, 15].
- En **Regresión Logística**, `scikit-learn` multiplica la suma de pérdidas por $C$. Al dividir la pérdida global entre $N \cdot C$, la penalización $L_2$ resulta en $\frac{1}{2 N C} \Vert{}w\Vert{}_2^2$, lo que exige fijar $C = \frac{1}{N \lambda_2}$[cite: 13, 15].

In [13]:
# ==============================================================================
# BENCHMARKING Y CONTRASTE NUMÉRICO CON SCIKIT-LEARN (PROBLEMA 4c)
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression

def compute_objective_value(model_name, X, y, w, b, l1_reg=0.0, l2_reg=0.0):
    """
    Calcula el valor exacto de la función objetivo F(w, b) bajo las convenciones del taller.
    """
    N = X.shape[0]
    z = X @ w + b

    if "Logistic" in model_name:
        # Pérdida logística estable por Log-Sum-Exp
        loss_i = np.maximum(0.0, z) - y * z + np.log(1.0 + np.exp(-np.abs(z)))
        j_val = np.mean(loss_i)
    else:
        # Pérdida residual de regresión J_reg
        res = z - y
        j_val = (1.0 / (2.0 * N)) * np.sum(res ** 2)

    penalty = (l1_reg * np.sum(np.abs(w))) + ((l2_reg / 2.0) * np.sum(w ** 2))
    return j_val + penalty


def benchmark_vs_sklearn(X, y, loss_type="linear", l1_reg=0.0, l2_reg=0.0):
    """
    Ejecuta la comparación entre las implementaciones en NumPy y scikit-learn.
    """
    N, D = X.shape
    results = {}

    if loss_type == "linear" and l1_reg == 0.0 and l2_reg == 0.0:
        model_title = "OLS (Mínimos Cuadrados)"
        # NumPy
        np_model = LinearRegressionClosedFormNumPy(l2_reg=0.0).fit(X, y)
        preds_np = np_model.predict(X)

        # sklearn
        sk_model = LinearRegression(fit_intercept=True).fit(X, y)
        preds_sk = sk_model.predict(X)

    elif loss_type == "linear" and l2_reg > 0.0:
        model_title = f"Ridge (lambda_2={l2_reg})"
        # Equivalencia sklearn: alpha = N * lambda_2
        alpha_sk = N * l2_reg

        np_model = LinearRegressionClosedFormNumPy(l2_reg=l2_reg).fit(X, y)
        preds_np = np_model.predict(X)

        sk_model = Ridge(alpha=alpha_sk, fit_intercept=True, solver='lbfgs' if hasattr(Ridge, 'lbfgs') else 'auto', tol=1e-10).fit(X, y)
        preds_sk = sk_model.predict(X)

    elif loss_type == "linear" and l1_reg > 0.0:
        model_title = f"Lasso (lambda_1={l1_reg})"
        # Equivalencia sklearn: alpha = lambda_1
        alpha_sk = l1_reg

        np_model = LassoProximalNumPy(l1_reg=l1_reg, eta=0.01, max_iter=20000, tol=1e-8).fit(X, y)
        preds_np = np_model.predict(X)

        sk_model = Lasso(alpha=alpha_sk, fit_intercept=True, max_iter=20000, tol=1e-8).fit(X, y)
        preds_sk = sk_model.predict(X)

    elif loss_type == "logistic":
        model_title = f"Regresión Logística L2 (lambda_2={l2_reg})"
        # Equivalencia sklearn: C = 1 / (N * lambda_2)
        C_sk = 1.0 / (N * l2_reg)

        np_model = GradientDescentLinearNumPy(loss_type="logistic", l2_reg=l2_reg, eta=0.2, max_iter=15000, tol=1e-8).fit(X, y)
        preds_np = np_model.predict(X)
        probs_np = np_model.predict_proba(X)[:, 1]

        sk_model = LogisticRegression(C=C_sk, penalty='l2', solver='lbfgs', fit_intercept=True, max_iter=15000, tol=1e-8).fit(X, y)
        preds_sk = sk_model.predict(X)
        probs_sk = sk_model.predict_proba(X)[:, 1]

    # Cálculo de métricas comparativas
    obj_np = compute_objective_value(model_title, X, y, np_model.w_, np_model.b_, l1_reg, l2_reg)
    obj_sk = compute_objective_value(model_title, X, y, sk_model.coef_.ravel(), float(sk_model.intercept_), l1_reg, l2_reg)

    diff_obj = np.abs(obj_np - obj_sk)
    rel_err_preds = np.linalg.norm(preds_np - preds_sk) / max(1.0, np.linalg.norm(preds_sk))

    results = {
        "Modelo": model_title,
        "N (filas)": N,
        "Objetivo NumPy": obj_np,
        "Objetivo sklearn": obj_sk,
        "|Diff Objetivo|": diff_obj,
        "Error Relativo Preds": rel_err_preds,
    }

    if loss_type == "logistic":
        max_diff_probs = np.max(np.abs(probs_np - probs_sk))
        results["Max Diff Probs"] = max_diff_probs
    else:
        results["Max Diff Probs"] = N/A_VAL if 'N/A_VAL' in locals() else "-"

    return results

In [20]:
# ==============================================================================
# RECALCULO DE LA EQUIVALENCIA lambda <-> alpha/C AL CAMBIAR EL NUMERO DE FILAS N
# ==============================================================================

def run_benchmarks_varying_N(seed=2026, N_values=(80, 200, 500), D=6):
    """
    Repite el contraste NumPy vs scikit-learn para Ridge, Lasso y Regresion
    Logistica con distintos tamanos de muestra N, mostrando explicitamente
    que la correspondencia alpha = N*lambda2 (Ridge) y C = 1/(N*lambda2)
    (Logistica) debe recalcularse en cada caso para mantener la equivalencia
    de los objetivos.
    """
    all_rows = []

    for N in N_values:
        rng = np.random.default_rng(seed)
        X_reg = rng.standard_normal((N, D))
        true_w = np.array([2.0, -1.5, 0.0, 0.8, 0.0, -0.5])
        y_reg = X_reg @ true_w + 0.5 + rng.standard_normal(N) * 0.1
        y_cls = (X_reg @ true_w + rng.standard_normal(N) * 0.2 > 0).astype(np.float64)

        for row in [
            benchmark_vs_sklearn(X_reg, y_reg, loss_type="linear", l1_reg=0.0, l2_reg=0.05),
            benchmark_vs_sklearn(X_reg, y_reg, loss_type="linear", l1_reg=0.02, l2_reg=0.0),
            benchmark_vs_sklearn(X_reg, y_cls, loss_type="logistic", l1_reg=0.0, l2_reg=0.1),
        ]:
            row["N"] = N
            all_rows.append(row)

    df_varying_N = pd.DataFrame(all_rows)
    cols_order = ["N", "Modelo", "Objetivo NumPy", "Objetivo sklearn",
                  "|Diff Objetivo|", "Error Relativo Preds", "Max Diff Probs"]
    df_varying_N = df_varying_N[cols_order]

    print("=== CONTRASTE NUMPY VS SCIKIT-LEARN AL VARIAR N ===")
    print("(la equivalencia alpha = N*lambda2 y C = 1/(N*lambda2) se recalcula en cada fila)\n")
    display(df_varying_N)

    print("\n--- VERIFICACION DE CRITERIO DE ACEPTACION (Discrepancia < 1e-4) POR CADA N ---")
    for _, row in df_varying_N.iterrows():
        status = "CUMPLIDO" if row["Error Relativo Preds"] < 1e-4 and row["|Diff Objetivo|"] < 1e-4 else "REVISAR"
        print(f"-> N={row['N']:>4} | {row['Modelo']}: "
              f"Error Rel = {row['Error Relativo Preds']:.2e} | "
              f"Diff Obj = {row['|Diff Objetivo|']:.2e} -> {status}")

    return df_varying_N

df_N_check = run_benchmarks_varying_N()

/tmp/ipykernel_2904/437254410.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  obj_sk = compute_objective_value(model_title, X, y, sk_model.coef_.ravel(), float(sk_model.intercept_), l1_reg, l2_reg)
/tmp/ipykernel_2904/437254410.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  obj_sk = compute_objective_value(model_title, X, y, sk_model.coef_.ravel(), float(sk_model.intercept_), l1_reg, l2_reg)


=== CONTRASTE NUMPY VS SCIKIT-LEARN AL VARIAR N ===
(la equivalencia alpha = N*lambda2 y C = 1/(N*lambda2) se recalcula en cada fila)



/tmp/ipykernel_2904/437254410.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  obj_sk = compute_objective_value(model_title, X, y, sk_model.coef_.ravel(), float(sk_model.intercept_), l1_reg, l2_reg)


,N,Modelo,Objetivo NumPy,Objetivo sklearn,|Diff Objetivo|,Error Relativo Preds,Max Diff Probs
0,80,Ridge (lambda_2=0.05),0.1733,0.1733,0.0000,0.0000,-
1,80,Lasso (lambda_1=0.02),0.1003,0.1003,0.0000,0.0000,-
2,80,Regresión Logística L2 (lambda_2=0.1),0.7454,0.7454,0.0000,0.0000,0.0000
3,200,Ridge (lambda_2=0.05),0.1743,0.1743,0.0000,0.0000,-
4,200,Lasso (lambda_1=0.02),0.1001,0.1001,0.0000,0.0000,-
5,200,Regresión Logística L2 (lambda_2=0.1),0.7812,0.7812,0.0000,0.0000,0.0000
6,500,Ridge (lambda_2=0.05),0.1741,0.1741,0.0000,0.0000,-
7,500,Lasso (lambda_1=0.02),0.0998,0.0998,0.0000,0.0000,-
8,500,Regresión Logística L2 (lambda_2=0.1),0.8406,0.8406,0.0000,0.0000,0.0000



--- VERIFICACION DE CRITERIO DE ACEPTACION (Discrepancia < 1e-4) POR CADA N ---
-> N=  80 | Ridge (lambda_2=0.05): Error Rel = 1.10e-15 | Diff Obj = 2.78e-17 -> CUMPLIDO
-> N=  80 | Lasso (lambda_1=0.02): Error Rel = 4.53e-09 | Diff Obj = 5.55e-17 -> CUMPLIDO
-> N=  80 | Regresión Logística L2 (lambda_2=0.1): Error Rel = 0.00e+00 | Diff Obj = 4.45e-09 -> CUMPLIDO
-> N= 200 | Ridge (lambda_2=0.05): Error Rel = 8.10e-16 | Diff Obj = 2.78e-17 -> CUMPLIDO
-> N= 200 | Lasso (lambda_1=0.02): Error Rel = 4.14e-09 | Diff Obj = 5.55e-17 -> CUMPLIDO
-> N= 200 | Regresión Logística L2 (lambda_2=0.1): Error Rel = 0.00e+00 | Diff Obj = 6.09e-08 -> CUMPLIDO
-> N= 500 | Ridge (lambda_2=0.05): Error Rel = 7.54e-16 | Diff Obj = 0.00e+00 -> CUMPLIDO
-> N= 500 | Lasso (lambda_1=0.02): Error Rel = 3.91e-09 | Diff Obj = 5.55e-17 -> CUMPLIDO
-> N= 500 | Regresión Logística L2 (lambda_2=0.1): Error Rel = 0.00e+00 | Diff Obj = 6.60e-08 -> CUMPLIDO


In [14]:
# ==============================================================================
# EJECUCIÓN DEL BENCHMARK SOBRE UN DATASET BIEN CONDICIONADO
# ==============================================================================

def run_all_benchmarks(seed=2026):
    np.random.seed(seed)
    N, D = 200, 6

    # Generar dataset sintético bien condicionado
    X_reg = np.random.randn(N, D)
    true_w = np.array([2.0, -1.5, 0.0, 0.8, 0.0, -0.5])
    y_reg = X_reg @ true_w + 0.5 + np.random.randn(N) * 0.1

    y_cls = (X_reg @ true_w + np.random.randn(N) * 0.2 > 0).astype(np.float64)

    benchmarks = []

    # 1. OLS
    benchmarks.append(benchmark_vs_sklearn(X_reg, y_reg, loss_type="linear", l1_reg=0.0, l2_reg=0.0))

    # 2. Ridge (l2 = 0.05)
    benchmarks.append(benchmark_vs_sklearn(X_reg, y_reg, loss_type="linear", l1_reg=0.0, l2_reg=0.05))

    # 3. Lasso (l1 = 0.02)
    benchmarks.append(benchmark_vs_sklearn(X_reg, y_reg, loss_type="linear", l1_reg=0.02, l2_reg=0.0))

    # 4. Logistic Regression L2 (l2 = 0.1)
    benchmarks.append(benchmark_vs_sklearn(X_reg, y_cls, loss_type="logistic", l1_reg=0.0, l2_reg=0.1))

    # Construir tabla comparativa de resultados
    df_results = pd.DataFrame(benchmarks)

    # Formatear números para lectura clara
    pd.set_option('display.float_format', lambda x: f'{x:.3e}' if abs(x) < 1e-2 else f'{x:.6f}')

    print("=== TABLA COMPARATIVA: NUMPY VS SCIKIT-LEARN ===")
    display(df_results)

    # Verificación de meta de discrepancia relativa < 1e-4
    print("\n--- VERIFICACIÓN DE CRITERIO DE ACEPTACIÓN (Discrepancia < 1e-4) ---")
    for idx, row in df_results.iterrows():
        err_p = row["Error Relativo Preds"]
        diff_o = row["|Diff Objetivo|"]
        status = "CUMPLIDO (< 1e-4)" if err_p < 1e-4 and diff_o < 1e-4 else "REVISAR"
        print(f"-> {row['Modelo']}: Error Relativo = {err_p:.2e} | Diff Obj = {diff_o:.2e} -> {status}")

run_all_benchmarks()

=== TABLA COMPARATIVA: NUMPY VS SCIKIT-LEARN ===


/tmp/ipykernel_2904/437254410.py:83: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  obj_sk = compute_objective_value(model_title, X, y, sk_model.coef_.ravel(), float(sk_model.intercept_), l1_reg, l2_reg)


,Modelo,N (filas),Objetivo NumPy,Objetivo sklearn,|Diff Objetivo|,Error Relativo Preds,Max Diff Probs
0,OLS (Mínimos Cuadrados),200,5.398e-03,5.398e-03,8.674e-19,9.468e-16,-
1,Ridge (lambda_2=0.05),200,0.174091,0.174091,2.776e-17,1.380e-15,-
2,Lasso (lambda_1=0.02),200,0.100387,0.100387,5.551e-17,3.900e-09,-
3,Regresión Logística L2 (lambda_2=0.1),200,0.675923,0.675923,5.586e-08,0.000e+00,1.282e-08



--- VERIFICACIÓN DE CRITERIO DE ACEPTACIÓN (Discrepancia < 1e-4) ---
-> OLS (Mínimos Cuadrados): Error Relativo = 9.47e-16 | Diff Obj = 8.67e-19 -> CUMPLIDO (< 1e-4)
-> Ridge (lambda_2=0.05): Error Relativo = 1.38e-15 | Diff Obj = 2.78e-17 -> CUMPLIDO (< 1e-4)
-> Lasso (lambda_1=0.02): Error Relativo = 3.90e-09 | Diff Obj = 5.55e-17 -> CUMPLIDO (< 1e-4)
-> Regresión Logística L2 (lambda_2=0.1): Error Relativo = 0.00e+00 | Diff Obj = 5.59e-08 -> CUMPLIDO (< 1e-4)


In [18]:
# ==============================================================================
# PRUEBA 5: INVARIANCIA DE PREDICT_PROBA Y LOG-LOSS RESPECTO AL UMBRAL TAU
# ==============================================================================

def test_tau_invariance(seed=2026):
    """
    Verifica que cambiar el umbral de decision tau NO altera predict_proba
    ni la perdida logaritmica (log-loss), ya que ambas dependen solo de z = Xw + b,
    nunca del umbral usado para binarizar la decision final.
    """
    np.random.seed(seed)
    N, D = 60, 4
    X = np.random.randn(N, D)
    y = (np.random.randn(N) > 0).astype(np.float64)

    print("--- PRUEBA 5: INVARIANCIA RESPECTO A TAU (UMBRAL DE DECISION) ---")

    model = GradientDescentLinearNumPy(loss_type="logistic", l2_reg=0.1, eta=0.1, max_iter=1000)
    model.fit(X, y)

    probs = model.predict_proba(X)[:, 1]  # p(y=1|x), fijas una vez entrenado el modelo

    # Log-loss (equivalente a J_log) calculada directamente desde las probabilidades
    eps = 1e-12
    log_loss_fixed = -np.mean(y * np.log(probs + eps) + (1 - y) * np.log(1 - probs + eps))

    taus = [0.1, 0.3, 0.5, 0.7, 0.9]
    probs_snapshots = []
    log_losses = []
    decisions = []

    for tau in taus:
        # predict_proba no debe depender de tau: se recalcula y se compara
        probs_tau = model.predict_proba(X)[:, 1]
        probs_snapshots.append(probs_tau)

        # la decision binaria SI depende de tau
        y_hat_tau = (probs_tau >= tau).astype(int)
        decisions.append(y_hat_tau)

        # la log-loss se calcula sobre las probabilidades, nunca sobre la decision binaria
        ll_tau = -np.mean(y * np.log(probs_tau + eps) + (1 - y) * np.log(1 - probs_tau + eps))
        log_losses.append(ll_tau)

    # Verificacion 1: las probabilidades son identicas para todo tau
    max_diff_probs = max(np.max(np.abs(p - probs)) for p in probs_snapshots)

    # Verificacion 2: la log-loss es identica para todo tau
    max_diff_logloss = max(abs(ll - log_loss_fixed) for ll in log_losses)

    # Verificacion 3: las decisiones binarias SI cambian con tau (contraste de control)
    n_distinct_decision_sets = len(set(tuple(d) for d in decisions))

    print(f"Max diff en predict_proba al variar tau: {max_diff_probs:.2e} (esperado = 0.0)")
    print(f"Max diff en log-loss al variar tau:      {max_diff_logloss:.2e} (esperado = 0.0)")
    print(f"Numero de vectores de decision distintos entre los {len(taus)} valores de tau: "
          f"{n_distinct_decision_sets} (se espera > 1, confirma que tau si afecta la decision)")

    passed = (max_diff_probs < 1e-14) and (max_diff_logloss < 1e-14) and (n_distinct_decision_sets > 1)
    print(f"Resultado Prueba 5: {'PASÓ CORRECTAMENTE' if passed else 'FALLÓ'}")

test_tau_invariance()

--- PRUEBA 5: INVARIANCIA RESPECTO A TAU (UMBRAL DE DECISION) ---
Max diff en predict_proba al variar tau: 0.00e+00 (esperado = 0.0)
Max diff en log-loss al variar tau:      0.00e+00 (esperado = 0.0)
Numero de vectores de decision distintos entre los 5 valores de tau: 5 (se espera > 1, confirma que tau si afecta la decision)
Resultado Prueba 5: PASÓ CORRECTAMENTE


### Discusión Metodológica: Comparación de Coeficientes y Convergencia

#### 1. ¿Por qué comparar coeficientes uno a uno no es una verificación suficiente cuando la solución no es única?
Cuando la matriz de diseño $X$ presenta **colinealidad perfecta** (p. ej. columnas duplicadas) o el número de características excede la cantidad de muestras ($D > N$), la solución de mínimos cuadrados o Lasso **no es única**[cite: 13, 14]:
- **Subespacio de minimizadores:** Existe una infinidad de vectores de parámetros $\hat{\theta}$ que alcanzan exactamente el mismo valor mínimo global de la función de costo[cite: 13, 14].
- **Criterio del algoritmo:** Algoritmos distintos (por ejemplo, resolución mediante la pseudoinversa SVD vs. eliminación gaussiana o descenso de coordenadas) convergen hacia puntos geométricos diferentes dentro del mismo subespacio óptimo.
- **Consecuencia:** Comparar los vectores $\hat{w}_{\text{numpy}}$ y $\hat{w}_{\text{sklearn}}$ elemento por elemento indicaría falsamente una discrepancia masiva, aun cuando ambos modelos produzcan **predicciones idénticas** y el **mismo valor de función objetivo**. Por esta razón, el criterio de validación principal debe basarse en el error relativo de predicción y en el valor del objetivo $F(w,b)$[cite: 15].

#### 2. Documentación de Discrepancias de Convergencia por Algoritmo
- **Métodos de Optimizador:** `GradientDescentLinearNumPy` utiliza un tamaño de paso (learning rate) $\eta$ fijo, mientras que `LogisticRegression` de `scikit-learn` emplea el algoritmo cuasi-Newton `L-BFGS`, que construye una aproximación de la matriz Hessiana para dar pasos adaptativos acelerados.
- **Diferencia de Criterio de Parada:** En Lasso, la implementación proximal evalúa la norma del residuo de la actualización $\frac{\Vert{}\theta^+ - \theta\Vert{}_2}{\eta} < \text{tol}$, mientras que `scikit-learn` evalúa la máxima violación de las condiciones KKT o la variación del objetivo entre iteraciones, lo que puede causar pequeñas variaciones en la última cifra decimal[cite: 15].

# Problema 5. Simulación de Sesgo, Varianza y Selección

---

## 1. Contexto y Diseño Experimental (Inciso a)

El objetivo de este experimento es estudiar el comportamiento empírico de las estimaciones lineales (OLS, Ridge y Lasso) frente al *tradeoff* de sesgo-varianza y la capacidad de selección de características bajo presencia de colinealidad en bloque.

### Configuración de Parámetros
* **Tamaño muestral:** $N = 120$ observaciones de entrenamiento y $M = 1000$ entradas de auditoría sintética $X_*$.
* **Dimensiones:** $D = 20$ características distribuidas en 4 bloques de 5 variables cada uno.
* **Estructura de Covarianza ($\Sigma_\rho$):** Para cada bloque $k \in \{1, 2, 3, 4\}$, la covarianza entre $x_i$ y $x_j$ es $\text{Cov}(x_i, x_j) = 1$ si $i=j$ y $\rho$ si $i \neq j$. Entre bloques distintos la correlación es nula ($0$). Se analizan dos escenarios: $\rho = 0.0$ (variables independientes) y $\rho = 0.9$ (alta correlación intrabloque).
* **Soporte Verdadero ($S^*$):** Únicamente la primera variable de cada bloque posee señal estructural:
  $$w^* = [3.0, 0, 0, 0, 0, \; -2.0, 0, 0, 0, 0, \; 1.5, 0, 0, 0, 0, \; -1.0, 0, 0, 0, 0]^\top$$
  por lo que $S^* = \{1, 6, 11, 16\}$ (índices Python `[0, 5, 10, 15]`).
* **Réplicas de Respuestas:** Se mantienen $X$ y $X_*$ fijos y se generan $R = 40$ realizaciones independientes de respuesta $y^{(r)} = X w^* + \epsilon^{(r)}$ con ruido $\epsilon^{(r)} \sim \mathcal{N}(0, I_N)$.
* **Condiciones Especiales:** Se fija el intercepto $b=0$ y no se aplica estandarización, garantizando escalas poblacionales idénticas por construcción para contrastar directamente los resultados analíticos del Problema 1c.

---

## 2. Descomposición Sesgo-Varianza y Verificación Teórica (Inciso b)

### Marco Matemático
Dado $f_m^* = x_{*m}^\top w^*$ y $\hat{f}_{rm} = x_{*m}^\top \hat{w}^{(r)}$, definimos el promedio sobre las $R$ réplicas como $\overline{f}_m = \frac{1}{R} \sum_{r=1}^R \hat{f}_{rm}$. Las componentes se estiman como:
* **Sesgo al cuadrado:** $\hat{B}^2 = \frac{1}{M} \sum_{m=1}^M (\overline{f}_m - f_m^*)^2$
* **Varianza:** $\hat{V} = \frac{1}{M} \sum_{m=1}^M \left[ \frac{1}{R} \sum_{r=1}^R (\hat{f}_{rm} - \overline{f}_m)^2 \right]$
* **Error Funcional Total:** $\hat{E}_f = \frac{1}{MR} \sum_{m=1}^M \sum_{r=1}^R (\hat{f}_{rm} - f_m^*)^2$

### Demostración de la Identidad Exacta $\hat{E}_f = \hat{B}^2 + \hat{V}$
Sumando y restando $\overline{f}_m$ dentro del término cuadrático:
$$(\hat{f}_{rm} - f_m^*)^2 = (\hat{f}_{rm} - \overline{f}_m)^2 + (\overline{f}_m - f_m^*)^2 + 2(\hat{f}_{rm} - \overline{f}_m)(\overline{f}_m - f_m^*)$$
Al promediar sobre las $R$ réplicas, el término cruzado se anula exactamente puesto que $\sum_{r=1}^R (\hat{f}_{rm} - \overline{f}_m) = R \overline{f}_m - R \overline{f}_m = 0$. Promediando sobre las $M$ observaciones de auditoría se obtiene la descomposición exacta $\hat{E}_f = \hat{B}^2 + \hat{V}$.

* **Factor $1/R$:** Representa el estimador muestral insesgado de la varianza sobre la distribución empírica de las $R$ respuestas simuladas.
* **Adición de $\sigma^2 = 1$ para Respuestas Ruidosas:** $\hat{E}_f$ mide el error de estimación de la **función latente pura** $f^* = X_* w^*$. Para predecir una nueva observación ruidosa $y_* = f^* + \epsilon_*$, la varianza irreducible del proceso estocástico $\sigma^2 = \text{Var}(\epsilon_*) = 1$ se adiciona de forma independiente, resultando en $\text{MSE}_{\text{pred}} = \hat{E}_f + \sigma^2 = \hat{B}^2 + \hat{V} + 1$.



### Contraste Teórico vs. Empírico en Ridge (Fórmulas Problema 1c)
Se evaluó la coincidencia entre los momentos empíricos obtenidos sobre las $R=40$ réplicas y los momentos teóricos condicionales a $X$:
$$\mathbb{E}[\hat{w}_{\lambda_2} \mid X] = (X^\top X + N \lambda_2 I_D)^{-1} X^\top X w^*$$
$$\text{Cov}(\hat{w}_{\lambda_2} \mid X) = \sigma^2 (X^\top X + N \lambda_2 I_D)^{-1} X^\top X (X^\top X + N \lambda_2 I_D)^{-1}$$

Las diferencias máximas observadas entre los valores esperados analíticos y los promedios muestrales son inferiores a $0.025$ para $\rho=0$ y diminutas en magnitud conforme aumenta $\lambda_2$, confirmando que las desviaciones responden de manera exclusiva al tamaño finito de muestra $R=40$.

---

## 3. Análisis de Selección de Variables con Lasso (Inciso c)

Se aplicó una tolerancia explícita $\text{tol} = 10^{-4}$ para determinar si un coeficiente es no nulo ($\vert{}\hat{w}_j\vert{} > 10^{-4}$).




In [15]:
# ==============================================================================
# PROBLEMA 5a: GENERACIÓN DEL EXPERIMENTO SINTÉTICO Y CONTROL DE SEMILLAS
# ==============================================================================

import numpy as np

def build_block_covariance(D=20, block_size=5, rho=0.0):
    """
    Construye la matriz de covarianza en bloques Sigma_rho (D x D).
    """
    Sigma = np.zeros((D, D), dtype=np.float64)
    n_blocks = D // block_size
    for b in range(n_blocks):
        start_idx = b * block_size
        end_idx = start_idx + block_size
        block = (1.0 - rho) * np.eye(block_size) + rho * np.ones((block_size, block_size))
        Sigma[start_idx:end_idx, start_idx:end_idx] = block
    return Sigma


def generate_synthetic_experiment(master_seed=2026, N=120, M=1000, D=20, R=40):
    """
    Genera los escenarios de simulación sintética (rho = 0.0 y rho = 0.9)
    con semillas trazables para garantizar reproducibilidad exacta.
    """
    # Vector de parámetros verdaderos w*
    w_star = np.zeros(D, dtype=np.float64)
    w_star[0] = 3.0    # w_1*
    w_star[5] = -2.0   # w_6*
    w_star[10] = 1.5   # w_11*
    w_star[15] = -1.0  # w_16*

    # Generador de subsemillas deterministas desde la semilla maestra
    ss = np.random.SeedSequence(master_seed)
    sub_seeds = ss.generate_state(n_words=6)

    seed_X_rho0 = sub_seeds[0]
    seed_Xstar_rho0 = sub_seeds[1]
    seed_noise_rho0 = sub_seeds[2]

    seed_X_rho09 = sub_seeds[3]
    seed_Xstar_rho09 = sub_seeds[4]
    seed_noise_rho09 = sub_seeds[5]

    scenarios = {}

    for rho, s_X, s_Xstar, s_noise in [(0.0, seed_X_rho0, seed_Xstar_rho0, seed_noise_rho0),
                                      (0.9, seed_X_rho09, seed_Xstar_rho09, seed_noise_rho09)]:

        Sigma_rho = build_block_covariance(D=D, block_size=5, rho=rho)

        # 1. Generar X (N x D)
        rng_X = np.random.default_rng(s_X)
        X = rng_X.multivariate_normal(mean=np.zeros(D), cov=Sigma_rho, size=N)

        # 2. Generar X_* de auditoría (M x D)
        rng_Xstar = np.random.default_rng(s_Xstar)
        X_star = rng_Xstar.multivariate_normal(mean=np.zeros(D), cov=Sigma_rho, size=M)

        # 3. Generar R réplicas de ruido epsilon^(r) ~ N(0, I_N)
        rng_noise = np.random.default_rng(s_noise)
        epsilon_train = rng_noise.normal(loc=0.0, scale=1.0, size=(R, N))

        # 4. Señal pura f* = X w* y f_* = X_* w*
        f_train_pure = X @ w_star
        f_star_pure = X_star @ w_star

        # 5. R respuestas Ruidosas y^(r) = X w* + epsilon^(r)
        Y_train_replicas = f_train_pure + epsilon_train  # Matriz (R x N)

        scenarios[rho] = {
            "rho": rho,
            "Sigma": Sigma_rho,
            "X": X,
            "X_star": X_star,
            "f_train_pure": f_train_pure,
            "f_star_pure": f_star_pure,
            "Y_train_replicas": Y_train_replicas,
            "w_star": w_star
        }

    return scenarios

# ==============================================================================
# AUDITORÍA Y VERIFICACIÓN DE LA GENERACIÓN SINTÉTICA
# ==============================================================================

sim_scenarios = generate_synthetic_experiment(master_seed=2026)

print("=== VERIFICACIÓN DE PARÁMETROS Y DIMENSIONES (PROBLEMA 5a) ===")
for rho, data in sim_scenarios.items():
    print(f"\n--- Escenario rho = {rho} ---")
    print(f"Dimensión X (Entrenamiento): {data['X'].shape}")
    print(f"Dimensión X_* (Auditoría):   {data['X_star'].shape}")
    print(f"Matriz de respuestas Y:     {data['Y_train_replicas'].shape} (R=40 réplicas x N=120 muestras)")

    # Verificar covarianza empírica del bloque 1
    cov_emp = np.cov(data['X'], rowvar=False)[:5, :5]
    print(f"Promedio de correlación dentro del Bloque 1 (Empírica): {np.mean(cov_emp[np.triu_indices(5, k=1)]):.4f}")

w_star_check = sim_scenarios[0.0]["w_star"]
non_zero_indices = np.where(w_star_check != 0)[0]
print(f"\nSoporte verdadero S*: Índices {non_zero_indices} -> Valores: {w_star_check[non_zero_indices]}")

=== VERIFICACIÓN DE PARÁMETROS Y DIMENSIONES (PROBLEMA 5a) ===

--- Escenario rho = 0.0 ---
Dimensión X (Entrenamiento): (120, 20)
Dimensión X_* (Auditoría):   (1000, 20)
Matriz de respuestas Y:     (40, 120) (R=40 réplicas x N=120 muestras)
Promedio de correlación dentro del Bloque 1 (Empírica): -0.0171

--- Escenario rho = 0.9 ---
Dimensión X (Entrenamiento): (120, 20)
Dimensión X_* (Auditoría):   (1000, 20)
Matriz de respuestas Y:     (40, 120) (R=40 réplicas x N=120 muestras)
Promedio de correlación dentro del Bloque 1 (Empírica): 0.9283

Soporte verdadero S*: Índices [ 0  5 10 15] -> Valores: [ 3.  -2.   1.5 -1. ]


In [16]:
# ==============================================================================
# PROBLEMA 5b: CÁLCULO Y VERIFICACIÓN DE SESGO, VARIANZA Y TEORÍA RIDGE
# ==============================================================================

import numpy as np
import pandas as pd

def run_bias_variance_experiment(sim_scenarios, lambda_grid=[1e-3, 1e-2, 1e-1, 1.0, 10.0]):
    """
    Ejecuta el experimento de sesgo-varianza sobre OLS, Ridge y Lasso.
    """
    results_bv = []
    theory_ridge_comparison = []

    for rho, sc in sim_scenarios.items():
        X = sc["X"]                      # (N x D)
        X_star = sc["X_star"]            # (M x D)
        Y_replicas = sc["Y_train_replicas"] # (R x N)
        f_star_pure = sc["f_star_pure"]  # (M,)
        w_star = sc["w_star"]            # (D,)

        R, N = Y_replicas.shape
        M, D = X_star.shape

        # --- 1. OLS (Referencia Sin Regularización) ---
        f_hat_ols = np.zeros((R, M), dtype=np.float64)
        w_hat_ols_replicas = np.zeros((R, D), dtype=np.float64)

        for r in range(R):
            y_r = Y_replicas[r]
            model_ols = LinearRegressionClosedFormNumPy(l2_reg=0.0).fit(X, y_r)
            w_hat_ols_replicas[r] = model_ols.w_
            f_hat_ols[r] = X_star @ model_ols.w_

        f_bar_ols = np.mean(f_hat_ols, axis=0)
        B2_ols = np.mean((f_bar_ols - f_star_pure) ** 2)
        V_ols = np.mean(np.var(f_hat_ols, axis=0, ddof=0))
        Ef_ols = np.mean((f_hat_ols - f_star_pure) ** 2)

        results_bv.append({
            "rho": rho, "Modelo": "OLS", "lambda": 0.0,
            "B^2 (Sesgo^2)": B2_ols, "V (Varianza)": V_ols,
            "E_f (Error Func)": Ef_ols, "B^2 + V": B2_ols + V_ols,
            "|Diff Descomp|": np.abs(Ef_ols - (B2_ols + V_ols))
        })

        # --- 2. RIDGE Y LASSO SOBRE LA GRILLA DE REGULARIZACIÓN ---
        for lam in lambda_grid:
            # --- RIDGE ---
            f_hat_ridge = np.zeros((R, M), dtype=np.float64)
            w_hat_ridge_replicas = np.zeros((R, D), dtype=np.float64)

            for r in range(R):
                y_r = Y_replicas[r]
                model_ridge = LinearRegressionClosedFormNumPy(l2_reg=lam).fit(X, y_r)
                w_hat_ridge_replicas[r] = model_ridge.w_
                f_hat_ridge[r] = X_star @ model_ridge.w_

            f_bar_ridge = np.mean(f_hat_ridge, axis=0)
            B2_ridge = np.mean((f_bar_ridge - f_star_pure) ** 2)
            V_ridge = np.mean(np.var(f_hat_ridge, axis=0, ddof=0))
            Ef_ridge = np.mean((f_hat_ridge - f_star_pure) ** 2)

            results_bv.append({
                "rho": rho, "Modelo": "Ridge", "lambda": lam,
                "B^2 (Sesgo^2)": B2_ridge, "V (Varianza)": V_ridge,
                "E_f (Error Func)": Ef_ridge, "B^2 + V": B2_ridge + V_ridge,
                "|Diff Descomp|": np.abs(Ef_ridge - (B2_ridge + V_ridge))
            })

            # --- VERIFICACIÓN TEÓRICA RIDGE (Fórmulas Prob. 1c) ---
            # E[w_hat|X] = (X^T X + N*lam*I)^(-1) X^T X w*
            M_inv = np.linalg.inv(X.T @ X + N * lam * np.eye(D))
            w_mean_theoretical = M_inv @ (X.T @ X) @ w_star
            cov_theoretical = M_inv @ (X.T @ X) @ M_inv # sigma^2 = 1.0

            w_mean_empirical = np.mean(w_hat_ridge_replicas, axis=0)
            cov_empirical = np.cov(w_hat_ridge_replicas, rowvar=False)

            diff_w_mean = np.max(np.abs(w_mean_empirical - w_mean_theoretical))
            diff_cov = np.max(np.abs(cov_empirical - cov_theoretical))

            theory_ridge_comparison.append({
                "rho": rho, "lambda": lam,
                "Max Diff E[w] (Emp vs Teor)": diff_w_mean,
                "Max Diff Cov(w) (Emp vs Teor)": diff_cov
            })

            # --- LASSO ---
            f_hat_lasso = np.zeros((R, M), dtype=np.float64)

            for r in range(R):
                y_r = Y_replicas[r]
                model_lasso = LassoProximalNumPy(l1_reg=lam, eta=0.005, max_iter=8000, tol=1e-6).fit(X, y_r)
                f_hat_lasso[r] = X_star @ model_lasso.w_

            f_bar_lasso = np.mean(f_hat_lasso, axis=0)
            B2_lasso = np.mean((f_bar_lasso - f_star_pure) ** 2)
            V_lasso = np.mean(np.var(f_hat_lasso, axis=0, ddof=0))
            Ef_lasso = np.mean((f_hat_lasso - f_star_pure) ** 2)

            results_bv.append({
                "rho": rho, "Modelo": "Lasso", "lambda": lam,
                "B^2 (Sesgo^2)": B2_lasso, "V (Varianza)": V_lasso,
                "E_f (Error Func)": Ef_lasso, "B^2 + V": B2_lasso + V_lasso,
                "|Diff Descomp|": np.abs(Ef_lasso - (B2_lasso + V_lasso))
            })

    return pd.DataFrame(results_bv), pd.DataFrame(theory_ridge_comparison)

# Ejecutar experimento
df_bv_results, df_ridge_theory = run_bias_variance_experiment(sim_scenarios)

print("=== TABLA 1: DESCOMPOSICIÓN SESGO-VARIANZA (PROBLEMA 5b) ===")
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
display(df_bv_results.head(12))

print("\n=== VERIFICACIÓN DE LA IDENTIDAD E_f = B^2 + V ===")
max_diff_identity = df_bv_results["|Diff Descomp|"].max()
print(f"Máxima discrepancia en la descomposición: {max_diff_identity:.2e} -> {'IDENTIDAD VERIFICADA EXACTAMENTE' if max_diff_identity < 1e-12 else 'REVISAR'}")

print("\n=== TABLA 2: CONTRASTE DE RIDGE TEÓRICO VS EMPÍRICO ===")
display(df_ridge_theory)

=== TABLA 1: DESCOMPOSICIÓN SESGO-VARIANZA (PROBLEMA 5b) ===


,rho,Modelo,lambda,B^2 (Sesgo^2),V (Varianza),E_f (Error Func),B^2 + V,|Diff Descomp|
0,0.000000,OLS,0.000000,0.002444,0.195322,0.197765,0.197765,0.000000
1,0.000000,Ridge,0.001000,0.002406,0.194727,0.197133,0.197133,0.000000
2,0.000000,Lasso,0.001000,0.002383,0.191632,0.194015,0.194015,0.000000
3,0.000000,Ridge,0.010000,0.003787,0.189515,0.193302,0.193302,0.000000
4,0.000000,Lasso,0.010000,0.002344,0.162061,0.164404,0.164404,0.000000
5,0.000000,Ridge,0.100000,0.149484,0.148501,0.297984,0.297984,0.000000
6,0.000000,Lasso,0.100000,0.038424,0.052812,0.091237,0.091237,0.000000
7,0.000000,Ridge,1.000000,4.065509,0.037969,4.103479,4.103479,0.000000
8,0.000000,Lasso,1.000000,3.353021,0.029720,3.382741,3.382741,0.000000
9,0.000000,Ridge,10.000000,13.951027,0.001306,13.952333,13.952333,0.000000



=== VERIFICACIÓN DE LA IDENTIDAD E_f = B^2 + V ===
Máxima discrepancia en la descomposición: 1.78e-15 -> IDENTIDAD VERIFICADA EXACTAMENTE

=== TABLA 2: CONTRASTE DE RIDGE TEÓRICO VS EMPÍRICO ===


,rho,lambda,Max Diff E[w] (Emp vs Teor),Max Diff Cov(w) (Emp vs Teor)
0,0.000000,0.001000,0.025733,0.004871
1,0.000000,0.010000,0.025429,0.004750
2,0.000000,0.100000,0.022882,0.003761
3,0.000000,1.000000,0.018623,0.000896
4,0.000000,10.000000,0.005692,0.000032
5,0.900000,0.001000,0.131385,0.045339
6,0.900000,0.010000,0.116885,0.033922
7,0.900000,0.100000,0.058382,0.007277
8,0.900000,1.000000,0.010255,0.000508
9,0.900000,10.000000,0.004795,0.000027


In [17]:
# ==============================================================================
# PROBLEMA 5c: SELECCIÓN DE VARIABLES CON LASSO, SOPORTE Y TRAYECTORIAS
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_lasso_selection(sim_scenarios, lambda_grid=[1e-3, 1e-2, 1e-1, 1.0, 10.0], zero_threshold=1e-4):
    """
    Evalúa la frecuencia de selección, precisión, recall y selección de bloques de Lasso.
    """
    S_star_indices = set([0, 5, 10, 15])  # Soporte verdadero (1-indexed: 1, 6, 11, 16)
    selection_metrics = []

    # Estructura para almacenar trayectorias promedio de coeficientes
    coeff_paths = {}

    for rho, sc in sim_scenarios.items():
        X = sc["X"]
        Y_replicas = sc["Y_train_replicas"]
        R, N = Y_replicas.shape
        M, D = sc["X_star"].shape

        coeff_paths[rho] = {lam: np.zeros((R, D)) for lam in lambda_grid}

        for lam in lambda_grid:
            selected_counts = np.zeros(D, dtype=int)
            precisions = []
            recalls = []

            for r in range(R):
                y_r = Y_replicas[r]
                model = LassoProximalNumPy(l1_reg=lam, eta=0.005, max_iter=8000, tol=1e-6).fit(X, y_r)
                w_hat = model.w_
                coeff_paths[rho][lam][r] = w_hat

                # Soporte estimado (coeficientes distintos de cero dada la tolerancia)
                S_hat_indices = set(np.where(np.abs(w_hat) > zero_threshold)[0])

                # Frecuencia de selección individual
                for idx in S_hat_indices:
                    selected_counts[idx] += 1

                # Métricas de soporte
                tp = len(S_hat_indices.intersection(S_star_indices))
                fp = len(S_hat_indices - S_star_indices)
                fn = len(S_star_indices - S_hat_indices)

                precision = tp / len(S_hat_indices) if len(S_hat_indices) > 0 else 0.0
                recall = tp / len(S_star_indices)

                precisions.append(precision)
                recalls.append(recall)

            freq_selection = selected_counts / R
            avg_precision = np.mean(precisions)
            avg_recall = np.mean(recalls)

            # Frecuencia promedio sobre variables verdaderas vs redundantes
            freq_true = np.mean(freq_selection[list(S_star_indices)])
            freq_noise = np.mean(freq_selection[list(set(range(D)) - S_star_indices)])

            selection_metrics.append({
                "rho": rho,
                "lambda": lam,
                "Precision Soporte": avg_precision,
                "Recall (Sensibilidad)": avg_recall,
                "Freq Promedio V. Verdaderas": freq_true,
                "Freq Promedio V. Irrelevantes": freq_noise,
                "Num Variables Activas (Prom)": np.mean([np.sum(np.abs(coeff_paths[rho][lam][r]) > zero_threshold) for r in range(R)])
            })

    return pd.DataFrame(selection_metrics), coeff_paths

# Ejecutar análisis de Lasso
zero_tol = 1e-4
df_lasso_selection, coeff_paths = analyze_lasso_selection(sim_scenarios, zero_threshold=zero_tol)

print(f"=== TABLA 3: SELECCIÓN DE SOPORTE Y MÉTRICAS DE RECUPERACIÓN (Tol = {zero_tol}) ===")
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
display(df_lasso_selection)

=== TABLA 3: SELECCIÓN DE SOPORTE Y MÉTRICAS DE RECUPERACIÓN (Tol = 0.0001) ===


,rho,lambda,Precision Soporte,Recall (Sensibilidad),Freq Promedio V. Verdaderas,Freq Promedio V. Irrelevantes,Num Variables Activas (Prom)
0,0.0000,0.0010,0.2024,1.0000,1.0000,0.9859,19.7750
1,0.0000,0.0100,0.2183,1.0000,1.0000,0.9000,18.4000
2,0.0000,0.1000,0.5349,1.0000,1.0000,0.2438,7.9000
3,0.0000,1.0000,1.0000,0.8688,0.8688,0.0000,3.4750
4,0.0000,10.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,0.9000,0.0010,0.2021,1.0000,1.0000,0.9875,19.8000
6,0.9000,0.0100,0.2461,1.0000,1.0000,0.7766,16.4250
7,0.9000,0.1000,0.4368,1.0000,1.0000,0.3406,9.4500
8,0.9000,1.0000,0.4217,0.7312,0.7312,0.2625,7.1250
9,0.9000,10.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## Discusión y Conclusiones del Experimento Sintético (Problema 5d)

### 1. ¿La regularización reduce simultáneamente sesgo y varianza?
**No.** La regularización no reduce ambos términos a la vez, sino que gestiona el compromiso (*tradeoff*) entre ellos:
* **Evidencia Empírica (Tabla 1):** Conforme se incrementa el hiperparámetro de penalización $\lambda$, la varianza $\hat{V}$ disminuye de forma estrictamente monótona hacia cero (de $0.1953$ en OLS a $0.0000$ en $\lambda=10$). En contraste, el sesgo al cuadrado $\hat{B}^2$ aumenta monótonamente (de $0.0024$ en OLS a $17.2858$ en $\lambda=10$).
* **Conclusión:** La regularización no elimina el sesgo; logra mejorar la estimación global reduciendo la varianza en una proporción sustancialmente mayor que el incremento en el sesgo para valores moderados de $\lambda$ (por ejemplo, con $\rho=0$ y $\lambda=0.10$, el error funcional total $\hat{E}_f$ cae a $0.0912$ frente a $0.1978$ de OLS).

---

### 2. ¿Mejor predicción implica recuperar el soporte correcto?
**No necesariamente.** Existe una tensión intrínseca entre la optimización de la capacidad predictiva (mínimo $\hat{E}_f$) y la recuperación exacta del soporte verdadero $S^* = \{1, 6, 11, 16\}$:
* **Evidencia Empírica ($\rho = 0.0$, Tabla 1 vs. Tabla 3):**
  * El mínimo error de estimación de la función se alcanza en $\lambda = 0.10$ ($\hat{E}_f = 0.0912$). Sin embargo, en este punto el modelo sobreestima el número de variables activas (retienen en promedio $7.9$ variables de las $20$), obteniendo una **Precisión de Soporte de solo $0.5349$**.
  * Para alcanzar **Precisión = $1.0000$** (eliminación total de variables irrelevantes en $\lambda = 1.00$), el Recall o Sensibilidad cae a $0.8688$ (comienza a eliminar coeficientes verdaderos más pequeños como $w_{16}^* = -1.0$), provocando que el error de predicción se degrade severamente a $\hat{E}_f = 3.3827$.
* **Conclusión:** La intensidad de regularización $\lambda$ que minimiza el error cuadrático conserva variables ruidosas para no perder señal, mientras que el $\lambda$ que aísla el soporte exacto contrae en exceso los coeficientes verdaderos.

---

### 3. ¿Cuándo una variable irrelevante estructuralmente funciona como sustituta de una relevante?
Ocurre en presencia de **alta colinealidad dentro del bloque ($\rho = 0.9$)**:
* **Evidencia Empírica ($\lambda = 1.00$, Tabla 3):**
  * En ausencia de correlación ($\rho = 0.0$), Lasso elimina por completo las variables irrelevantes (Frecuencia Promedio = $0.0000$, Precisión = $1.0000$).
  * Bajo alta correlación ($\rho = 0.9$), la Frecuencia Promedio de selección de variables estructuralmente irrelevantes sube a **$0.2625$** y la Precisión colapsa a **$0.4217$**.
* **Explicación Teórica:** La presencia de características altamente correlacionadas destruye la estricta convexidad de la función objetivo con penalización $L_1$ (demostrado en el Problema 2b). Esto genera un subespacio de óptimos o soluciones cercanas donde el algoritmo selecciona o intercambia inestablemente variables del mismo bloque, permitiendo que variables irrelevantes actúen como sustitutas de la variable activa relevante.